# Aula 2 — Whisper (Hugging Face), camada a camada

**Duração sugerida:** ~45 min
**Pré-requisito:** notebook 1 (Transformer simplificado). Tudo o que aparece aqui já foi implementado lá: atenção, multi-cabeça, residual, LayerNorm, FFN, encoder-decoder.
**Execução:** CPU é suficiente com `whisper-tiny` ou `whisper-base`. A primeira execução baixa o modelo (≈150 MB para o `tiny`); depois fica em cache.

### Roteiro

| # | Seção | Tempo |
|---|-------|-------|
| 1 | Visão geral: um Transformer encoder-decoder para **áudio** | 5 min |
| 2 | Do áudio ao espectrograma log-Mel (`(1, 80, 3000)`) | 7 min |
| 3 | Carregando o modelo e inspecionando a arquitetura | 4 min |
| 4 | **Encoder**, camada a camada (stem convolucional, posição, bloco pré-norm) | 10 min |
| 5 | **Decoder**, camada a camada (tokens, atenção cruzada, saída) | 10 min |
| 6 | Geração autorregressiva "na mão" × `model.generate` | 5 min |
| 7 | Atenção cruzada: onde o decoder "escuta" | 3 min |
| 8 | `pipeline` de alto nível e áudios longos | 1 min |

Fontes: Radford et al. (2022), *Robust Speech Recognition via Large-Scale Weak Supervision* (Figura 1, Tabela 1, seção 2); Wayne Yu (*Whisper: Robust Speech Recognition...*, Medium); *Speech Recognition using Transformers in Python* (The Python Code).

## 0. Preparação

Defina abaixo o modelo e o arquivo de áudio. Os tamanhos disponíveis (Tabela 1 do paper):

| Modelo | Camadas (enc = dec) | Largura `d_model` | Cabeças | Parâmetros |
|---|---|---|---|---|
| `openai/whisper-tiny` | 4 | 384 | 6 | 39 M |
| `openai/whisper-base` | 6 | 512 | 8 | 74 M |
| `openai/whisper-small` | 12 | 768 | 12 | 244 M |
| `openai/whisper-medium` | 24 | 1024 | 16 | 769 M |
| `openai/whisper-large` | 32 | 1280 | 20 | 1550 M |

Para aula em CPU: **tiny** (rápido) ou **base** (melhor em português). O **small** é bem melhor em português, mas fica lento ao vivo sem GPU.

In [ ]:
import os

# Evita a queda do kernel no Windows/conda (duas cópias da biblioteca OpenMP: PyTorch + numpy/MKL)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F
import transformers
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
from transformers.modeling_outputs import BaseModelOutput

warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")      # silencia aviso do Windows

# ------------------- PARÂMETROS DA AULA -------------------
MODEL_ID = "openai/whisper-tiny"      # ou "openai/whisper-base" / "openai/whisper-small"
AUDIO = "amostra_pt.wav"              # áudio em português (wav, mp3, m4a...), ideal com 5 a 25 s
# -----------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | dispositivo: {DEVICE}")

## 1. Visão geral

O Whisper é um **Transformer encoder-decoder padrão** (o mesmo do notebook 1), treinado com 680 mil horas de áudio rotulado de forma "fraca" pela internet, em cerca de 100 idiomas. O que muda é só a **entrada** (áudio, em vez de tokens) e a **interface do decoder** (tokens especiais que dizem *o que fazer*):

```
 áudio 16 kHz (30 s)
   │  log-Mel (janela 25 ms, passo 10 ms)
   ▼
 (80 × 3000)  ──► Conv1D(k=3) + GELU ──► Conv1D(k=3, stride 2) + GELU ──► (1500 × d_model)
                                                                              │ + posição SENOIDAL
                                                                              ▼
                                                        [ Bloco do encoder (pré-norm) × N ] ── LayerNorm
                                                                              │ memory (1500 × d_model)
                                                                              ▼  (K, V da atenção cruzada)
 <|startoftranscript|> <|pt|> <|transcribe|> <|notimestamps|> ──► Embedding + posição APRENDIDA
                                                        [ Bloco do decoder (pré-norm) × N ] ── LayerNorm
                                                                              ▼
                                                        Linear (pesos amarrados ao embedding) ──► próximo token
```

| | Transformer do notebook 1 | Whisper |
|---|---|---|
| Entrada do encoder | embedding de tokens | espectrograma + 2 convoluções |
| Posição no encoder | senoidal | senoidal |
| Posição no decoder | senoidal | **aprendida** |
| Normalização | pós-norm | **pré-norm** (+ LayerNorm final) |
| Ativação da FFN | ReLU | **GELU** |
| Saída | `Linear` própria | `Linear` com pesos **compartilhados** com o embedding |

## 2. Do áudio ao espectrograma log-Mel

Um modelo de áudio não lê a forma de onda crua (480 000 amostras para 30 s). A entrada é um **espectrograma log-Mel**:

1. reamostrar o áudio para **16 kHz** (mono);
2. **janelas de 25 ms** (400 amostras), deslocadas de **10 ms** (160 amostras), com a Transformada de Fourier de cada janela;
3. agrupar as frequências em **80 bandas Mel** (escala que imita a percepção humana);
4. tirar o **log** e normalizar.

O Whisper sempre trabalha com **janelas de exatamente 30 s**: áudios curtos são completados com silêncio, longos são cortados (ou fatiados pelo `pipeline`, seção 8). Resultado: `100 quadros/s × 30 s = 3000 quadros`.

In [ ]:
from scipy.signal import resample_poly

SR = 16_000

def carregar_audio(caminho, sr_alvo=SR):
    """Lê o áudio, converte para mono e reamostra para 16 kHz (float32)."""
    try:
        dados, sr = sf.read(caminho, dtype="float32", always_2d=True)
        dados = dados.mean(axis=1)                                    # estéreo -> mono
        if sr != sr_alvo:
            g = math.gcd(sr, sr_alvo)
            dados = resample_poly(dados, sr_alvo // g, sr // g)
        return dados.astype("float32")
    except Exception:                                                  # mp3/m4a etc.: usa o ffmpeg
        from transformers.pipelines.audio_utils import ffmpeg_read
        with open(caminho, "rb") as f:
            return ffmpeg_read(f.read(), sr_alvo)

**Não tem um áudio em português?** A célula abaixo só roda se `AUDIO` não existir: no Windows, gera um áudio com a voz do sistema (`pyttsx3`). Também vale gravar com o *Gravador de Voz* do Windows (arquivo `.m4a`: o `ffmpeg` do ambiente lê) e salvar na pasta do notebook.

In [ ]:
FRASE = ("Olá, bem-vindos à aula sobre transformers. "
         "O modelo Whisper transcreve a fala em texto, inclusive em português.")

if not os.path.exists(AUDIO):
    try:
        import pyttsx3                                                 # só no Windows (voz SAPI5)
        tts = pyttsx3.init()
        for v in tts.getProperty("voices"):
            if any(s in (v.name + v.id).lower() for s in ("portug", "brazil", "maria", "daniel")):
                tts.setProperty("voice", v.id)
                break
        tts.save_to_file(FRASE, AUDIO)
        tts.runAndWait()
        print("Áudio gerado em", AUDIO)
    except Exception as e:
        raise FileNotFoundError(f"Coloque um áudio em português em '{AUDIO}' (e rode de novo). Detalhe: {e}")
else:
    print("Usando o áudio existente:", AUDIO)

In [ ]:
audio = carregar_audio(AUDIO)
duracao = len(audio) / SR
print(f"{len(audio):,} amostras a {SR} Hz = {duracao:.1f} s | amplitude máx. {np.abs(audio).max():.2f}")

plt.figure(figsize=(9, 2.2))
plt.plot(np.arange(len(audio)) / SR, audio, lw=0.5)
plt.xlabel("tempo (s)"); plt.title("Forma de onda (16 kHz, mono)"); plt.tight_layout(); plt.show()

## 3. Carregando o modelo

Usamos `attn_implementation="eager"` para que o modelo consiga devolver os **pesos de atenção** (seção 7). Para só transcrever, o padrão é mais rápido.

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, attn_implementation="eager").to(DEVICE).eval()
tok = processor.tokenizer
cfg = model.config

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params / 1e6:.1f} M parâmetros")
print(f"  d_model = {cfg.d_model} | cabeças = {cfg.encoder_attention_heads} | camadas enc/dec = {cfg.encoder_layers}/{cfg.decoder_layers}")
print(f"  FFN = {cfg.encoder_ffn_dim} | bandas Mel = {cfg.num_mel_bins} | vocabulário = {cfg.vocab_size}")
print(f"  posições do encoder = {cfg.max_source_positions} | posições do decoder = {cfg.max_target_positions}")

In [ ]:
def resumo(modulo, nome="modelo", nivel=0, max_nivel=3):
    """Imprime a árvore de módulos (até max_nivel) com a contagem de parâmetros."""
    n = sum(p.numel() for p in modulo.parameters())
    print("  " * nivel + f"{nome}: {type(modulo).__name__}  ({n:,} parâmetros)")
    if nivel < max_nivel:
        for nome_filho, filho in modulo.named_children():
            if nome_filho == "layers" and nivel >= 2:                  # não repete as N camadas iguais
                print("  " * (nivel + 1) + f"layers: {len(filho)} × {type(filho[0]).__name__}")
                resumo(filho[0], "layers[0]", nivel + 2, max_nivel)
                continue
            resumo(filho, nome_filho, nivel + 1, max_nivel)

resumo(model, "modelo", max_nivel=4)

Repare na correspondência com o notebook 1:

- `encoder.layers[i]` = `EncoderBlock` (autoatenção + FFN); `decoder.layers[i]` = `DecoderBlock` (autoatenção + **`encoder_attn`** = atenção cruzada + FFN).
- `self_attn_layer_norm`, `encoder_attn_layer_norm`, `final_layer_norm` = as LayerNorms de cada subcamada.
- `fc1`/`fc2` = a FFN. `q_proj`, `k_proj`, `v_proj`, `out_proj` = $W^Q, W^K, W^V, W^O$ do multi-cabeça (note que `k_proj` **não tem bias**).

### 3.1 Áudio → `input_features`

In [ ]:
fe = processor.feature_extractor
print(f"janela = {fe.n_fft} amostras ({1000 * fe.n_fft / SR:.0f} ms) | passo = {fe.hop_length} amostras ({1000 * fe.hop_length / SR:.0f} ms)")
print(f"30 s = {fe.n_samples:,} amostras -> {fe.nb_max_frames} quadros Mel de {fe.feature_size} bandas")

inputs = fe(audio, sampling_rate=SR, return_tensors="pt")
input_features = inputs.input_features.to(DEVICE)
print("input_features:", tuple(input_features.shape), "= (lote, bandas Mel, quadros)")

plt.figure(figsize=(9, 3))
plt.imshow(input_features[0].cpu().numpy(), origin="lower", aspect="auto", extent=[0, 30, 0, 80], cmap="magma")
plt.axvline(duracao, color="w", ls="--", lw=1)
plt.xlabel("tempo (s)  (à direita da linha: preenchimento de silêncio até 30 s)"); plt.ylabel("banda Mel")
plt.title("Espectrograma log-Mel"); plt.colorbar(label="log-magnitude normalizada"); plt.tight_layout(); plt.show()

## 4. Encoder, camada a camada

Para provar que entendemos cada etapa, vamos **recalcular à mão** a saída de cada parte e comparar com o que o modelo calcula. Primeiro capturamos, com *hooks*, a entrada e a saída da primeira camada do encoder.

In [ ]:
def primeiro(x):
    """Camadas devolvem um tensor (versões novas) ou uma tupla (versões antigas)."""
    return x[0] if isinstance(x, tuple) else x

capt = {}
enc = model.model.encoder

def pre_hook(nome):
    def fn(modulo, args, kwargs):
        capt[nome + "_in"] = (args[0] if args else kwargs["hidden_states"]).detach()
    return fn

def post_hook(nome):
    def fn(modulo, args, kwargs, saida):
        capt[nome + "_out"] = primeiro(saida).detach()
    return fn

hooks = [
    enc.layers[0].register_forward_pre_hook(pre_hook("enc0"), with_kwargs=True),
    enc.layers[0].register_forward_hook(post_hook("enc0"), with_kwargs=True),
    enc.layers[-1].register_forward_hook(post_hook("encN"), with_kwargs=True),
]
with torch.no_grad():
    enc_out = enc(input_features).last_hidden_state              # (1, 1500, d_model)
for h in hooks:
    h.remove()
print("saída do encoder (a 'memory'):", tuple(enc_out.shape), "= (lote, 1500 quadros de 20 ms, d_model)")

### 4.1 Stem convolucional

Duas `Conv1d` de largura 3 com GELU. A **segunda tem stride 2**, que reduz 3000 → 1500 posições (cada posição passa a cobrir **20 ms**). Isso corta pela metade o custo quadrático da atenção.

| etapa | formato |
|---|---|
| `input_features` | `(1, 80, 3000)` |
| `conv1` (80 → d_model, k=3, stride 1) + GELU | `(1, d_model, 3000)` |
| `conv2` (d_model → d_model, k=3, stride 2) + GELU | `(1, d_model, 1500)` |
| `permute(0, 2, 1)` | `(1, 1500, d_model)` |

In [ ]:
with torch.no_grad():
    x = F.gelu(enc.conv1(input_features));  print("após conv1 + GELU:", tuple(x.shape))
    x = F.gelu(enc.conv2(x));               print("após conv2 + GELU:", tuple(x.shape))
    x = x.permute(0, 2, 1);                 print("após permute     :", tuple(x.shape))

### 4.2 Codificação posicional (fixa, senoidal)

Soma-se ao resultado do stem. Na versão do Whisper, os senos ocupam a **primeira metade** dos canais e os cossenos a **segunda** (concatenados), em vez de intercalados como no paper original; e o espaçamento das frequências usa `d/2 − 1`. A tabela é **fixa** (não treinável). Vamos reconstruí-la e comparar com os pesos do modelo:

In [ ]:
def sinusoides(comprimento, canais, max_timescale=10000):
    log_inc = math.log(max_timescale) / (canais // 2 - 1)
    inv = torch.exp(-log_inc * torch.arange(canais // 2))
    t = torch.arange(comprimento)[:, None] * inv[None, :]
    return torch.cat([t.sin(), t.cos()], dim=1)                      # (comprimento, canais)

pos_enc = enc.embed_positions.weight.detach().cpu()
print("tabela de posições do encoder:", tuple(pos_enc.shape), "| treinável?", enc.embed_positions.weight.requires_grad)
print("igual à fórmula senoidal?", torch.allclose(pos_enc, sinusoides(*pos_enc.shape), atol=1e-4))

with torch.no_grad():
    x = x + enc.embed_positions.weight[:x.size(1)]                   # soma da posição
entrada_camada0 = capt["enc0_in"]
print("stem + posição à mão == entrada da camada 0 do encoder?", torch.allclose(x, entrada_camada0, atol=1e-4))

### 4.3 Um bloco do encoder (pré-norm)

Diferença-chave para o notebook 1: a **LayerNorm vem antes** da subcamada, e o resíduo soma a entrada **sem normalizar**:

$$x \leftarrow x + \text{Attn}\big(\text{LN}(x)\big),\qquad x \leftarrow x + \text{FFN}\big(\text{LN}(x)\big)$$

Dentro de `WhisperAttention`, o escalonamento $1/\sqrt{d_k}$ é aplicado em $Q$ antes do produto. Abaixo reimplementamos a atenção usando apenas as projeções lineares do módulo.

In [ ]:
def atencao_na_mao(attn, x_q, x_kv, causal=False):
    """WhisperAttention recalculada: mesmas contas do notebook 1, usando as projeções do módulo."""
    B, Tq, D = x_q.shape
    Tk, h, dk = x_kv.size(1), attn.num_heads, attn.head_dim
    q = (attn.q_proj(x_q) * dk ** -0.5).view(B, Tq, h, dk).transpose(1, 2)     # (B, h, Tq, dk)
    k = attn.k_proj(x_kv).view(B, Tk, h, dk).transpose(1, 2)                   # (B, h, Tk, dk)
    v = attn.v_proj(x_kv).view(B, Tk, h, dk).transpose(1, 2)
    scores = q @ k.transpose(-2, -1)                                           # (B, h, Tq, Tk)
    if causal:
        bloqueio = torch.triu(torch.ones(Tq, Tk, dtype=torch.bool, device=x_q.device), diagonal=1)
        scores = scores.masked_fill(bloqueio, float("-inf"))
    pesos = F.softmax(scores, dim=-1)
    saida = (pesos @ v).transpose(1, 2).reshape(B, Tq, D)
    return attn.out_proj(saida), pesos

camada = enc.layers[0]
with torch.no_grad():
    h = capt["enc0_in"]
    r = h
    h = camada.self_attn_layer_norm(h)                               # 1) LayerNorm
    h, pesos_enc0 = atencao_na_mao(camada.self_attn, h, h)           # 2) autoatenção (sem máscara)
    h = r + h                                                        # 3) resíduo
    r = h
    h = camada.final_layer_norm(h)                                   # 4) LayerNorm
    h = camada.fc2(camada.activation_fn(camada.fc1(h)))              # 5) FFN: fc1 -> GELU -> fc2
    h = r + h                                                        # 6) resíduo

print("pesos de atenção da camada 0:", tuple(pesos_enc0.shape), "= (lote, cabeças, 1500, 1500)")
print("bloco 0 à mão == saída da camada 0 do modelo?", torch.allclose(h, capt["enc0_out"], atol=1e-3))
print(f"FFN: {camada.fc1.in_features} -> {camada.fc1.out_features} -> {camada.fc2.out_features} (expansão de {camada.fc1.out_features // camada.fc1.in_features}×)")

### 4.4 Pilha completa e LayerNorm final

As $N$ camadas se repetem (`encoder.layers`), seguidas de **uma LayerNorm final** (`encoder.layer_norm`, necessária porque o pré-norm deixa o fluxo residual sem normalizar). A saída `(1, 1500, d_model)` é a "memória" que o decoder consulta.

In [ ]:
with torch.no_grad():
    saida_final = enc.layer_norm(capt["encN_out"])
print("LayerNorm final(última camada) == saída do encoder?", torch.allclose(saida_final, enc_out, atol=1e-4))
print(f"o encoder produziu {enc_out.size(1)} vetores de {enc_out.size(2)} dimensões para {duracao:.1f} s de áudio "
      f"(1 vetor a cada 20 ms; só os primeiros ≈ {int(duracao / 0.02)} cobrem fala).")

## 5. Decoder, camada a camada

### 5.1 Tokens especiais: o "prompt" que define a tarefa

O decoder começa com um **prefixo** que diz ao modelo o que fazer. É assim que um único modelo faz transcrição, tradução, detecção de idioma e timestamps:

| posição | token | papel |
|---|---|---|
| 0 | `<|startoftranscript|>` | início |
| 1 | `<|pt|>` | **idioma** (um token por idioma, ≈100) |
| 2 | `<|transcribe|>` (ou `<|translate|>`) | **tarefa** |
| 3 | `<|notimestamps|>` | sem marcas de tempo (se omitido, o modelo emite `<|0.00|>` ...) |
| … | texto em BPE | transcrição |
| fim | `<|endoftext|>` | fim |

In [ ]:
prefixo_tokens = ["<|startoftranscript|>", "<|pt|>", "<|transcribe|>", "<|notimestamps|>"]
prefixo_ids = tok.convert_tokens_to_ids(prefixo_tokens)
eos_id = tok.eos_token_id
for t, i in zip(prefixo_tokens, prefixo_ids):
    print(f"{t:24s} id = {i}")
print(f"{'<|endoftext|>':24s} id = {eos_id}")

decoder_ids = torch.tensor([prefixo_ids], device=DEVICE)                      # (1, 4)
print("\ndecoder_input_ids:", tuple(decoder_ids.shape))

**Detecção de idioma** é o primeiro passo do decoder: com apenas `<|startoftranscript|>`, o modelo prevê *qual token de idioma vem a seguir*. Vamos olhar as probabilidades (precisa dos metadados do modelo real; se não houver, a célula avisa).

In [ ]:
try:
    lang_to_id = model.generation_config.lang_to_id                          # {"<|pt|>": id, ...}
    with torch.no_grad():
        lg = model(encoder_outputs=BaseModelOutput(last_hidden_state=enc_out),
                   decoder_input_ids=torch.tensor([[prefixo_ids[0]]], device=DEVICE), use_cache=False).logits[0, -1]
    ids_idioma = torch.tensor(list(lang_to_id.values()), device=lg.device)
    probs = F.softmax(lg[ids_idioma], dim=-1)
    top = probs.topk(3)
    nomes = {v: k for k, v in lang_to_id.items()}
    for p, i in zip(top.values, top.indices):
        print(f"{nomes[ids_idioma[i].item()]:8s} {p.item():.1%}")
except Exception as e:
    print("Detecção de idioma indisponível neste modelo:", type(e).__name__)

### 5.2 Embedding de tokens + posição **aprendida**

No decoder, a posição é uma tabela `nn.Embedding(448, d_model)` **treinada** (448 = `max_target_positions`), ao contrário da senoidal do encoder.

In [ ]:
dec = model.model.decoder
capt_dec = {}
hooks = [
    dec.layers[0].register_forward_pre_hook(
        lambda m, a, kw: capt_dec.__setitem__("dec0_in", (a[0] if a else kw["hidden_states"]).detach()), with_kwargs=True),
    dec.layers[0].register_forward_hook(
        lambda m, a, kw, out: capt_dec.__setitem__("dec0_out", primeiro(out).detach()), with_kwargs=True),
    dec.layers[-1].register_forward_hook(
        lambda m, a, kw, out: capt_dec.__setitem__("decN_out", primeiro(out).detach()), with_kwargs=True),
]
with torch.no_grad():
    dec_out = dec(input_ids=decoder_ids, encoder_hidden_states=enc_out, use_cache=False).last_hidden_state
for h in hooks:
    h.remove()

T = decoder_ids.size(1)
with torch.no_grad():
    emb = dec.embed_tokens(decoder_ids) + dec.embed_positions.weight[:T]      # embedding + posição
print("embed_tokens:", tuple(dec.embed_tokens.weight.shape), "= (vocabulário, d_model)")
print("embed_positions:", tuple(dec.embed_positions.weight.shape), "| treinável?", dec.embed_positions.weight.requires_grad)
print("embedding + posição à mão == entrada da camada 0 do decoder?", torch.allclose(emb, capt_dec["dec0_in"], atol=1e-4))

### 5.3 Um bloco do decoder: três subcamadas

1. **Autoatenção mascarada** (causal): cada token só vê os anteriores.
2. **Atenção cruzada** (`encoder_attn`): **Q** vem do decoder; **K e V** vêm da `memory` do encoder, com 1500 posições. É aqui que o texto "consulta" o áudio.
3. **FFN**.

Todas pré-norm, com resíduo.

In [ ]:
camada = dec.layers[0]
with torch.no_grad():
    h = capt_dec["dec0_in"]
    r = h; h = camada.self_attn_layer_norm(h)
    h, w_self = atencao_na_mao(camada.self_attn, h, h, causal=True)          # Q, K, V = texto (com máscara causal)
    h = r + h
    r = h; h = camada.encoder_attn_layer_norm(h)
    h, w_cross = atencao_na_mao(camada.encoder_attn, h, enc_out)             # Q = texto; K, V = áudio
    h = r + h
    r = h; h = camada.final_layer_norm(h)
    h = camada.fc2(camada.activation_fn(camada.fc1(h)))
    h = r + h

print("autoatenção     :", tuple(w_self.shape), "= (lote, cabeças, T, T)       -> triangular (máscara causal)")
print("atenção cruzada :", tuple(w_cross.shape), "= (lote, cabeças, T, 1500)    -> cada token olha para os 1500 quadros de áudio")
print("bloco 0 do decoder à mão == saída da camada 0 do modelo?", torch.allclose(h, capt_dec["dec0_out"], atol=1e-3))

### 5.4 LayerNorm final e projeção para o vocabulário

Depois das $N$ camadas: `decoder.layer_norm` e uma projeção linear `proj_out` de `d_model` para `vocab_size` (≈ 51 865 tokens). Os pesos de `proj_out` são **os mesmos** do embedding de entrada (*weight tying*, Press & Wolf, 2017): economiza parâmetros e alinha "entender" e "gerar" tokens.

In [ ]:
with torch.no_grad():
    h_final = dec.layer_norm(capt_dec["decN_out"])
    logits = model.proj_out(h_final)                                         # (1, T, vocab)
print("LayerNorm final == saída do decoder?", torch.allclose(h_final, dec_out, atol=1e-4))
print("logits:", tuple(logits.shape), "= (lote, T, vocabulário)")
print("proj_out compartilha os pesos de embed_tokens?", model.proj_out.weight.data_ptr() == dec.embed_tokens.weight.data_ptr())

prox = logits[0, -1].softmax(-1).topk(5)
print("\n5 tokens mais prováveis após o prefixo:")
for p, i in zip(prox.values, prox.indices):
    print(f"  {tok.convert_ids_to_tokens(i.item())!r:22s} {p.item():.1%}")

## 6. Geração autorregressiva "na mão" × `model.generate`

Igual ao `greedy_decode` do notebook 1: o **encoder roda uma vez**; o decoder roda a cada passo, e escolhemos o token mais provável até aparecer `<|endoftext|>` (sem cache de chaves/valores, para simplificar; o `generate` usa cache e roda muito mais rápido).

In [ ]:
@torch.no_grad()
def decodificar_na_mao(memoria, prefixo, max_tokens=120):
    ids = torch.tensor([prefixo], device=memoria.device)
    saida = BaseModelOutput(last_hidden_state=memoria)
    for passo in range(max_tokens):
        lg = model(encoder_outputs=saida, decoder_input_ids=ids, use_cache=False).logits[:, -1]
        prox = lg.argmax(-1, keepdim=True)
        ids = torch.cat([ids, prox], dim=1)
        if prox.item() == eos_id:
            break
    return ids[0]

t0 = time.time()
ids_mao = decodificar_na_mao(enc_out, prefixo_ids)
t_mao = time.time() - t0
texto_mao = tok.decode(ids_mao, skip_special_tokens=True).strip()
print(f"[na mão, {t_mao:.1f} s, {len(ids_mao) - len(prefixo_ids)} tokens gerados]")
print("primeiros tokens:", tok.convert_ids_to_tokens(ids_mao[len(prefixo_ids):].tolist())[:25])
print("texto :", texto_mao)

In [ ]:
# (requer o modelo real: `generate` usa a configuração de geração e o cache de K/V)
t0 = time.time()
with torch.no_grad():
    ids_gen = model.generate(input_features, language="pt", task="transcribe", max_new_tokens=120)
t_gen = time.time() - t0
texto_gen = processor.batch_decode(ids_gen, skip_special_tokens=True)[0].strip()
print(f"[generate, {t_gen:.1f} s]")
print("texto :", texto_gen)
print("\nmesmo texto da geração manual?", texto_gen == texto_mao)

Se os textos diferirem, é esperado em pequenos detalhes: o `generate` aplica regras extras (suprime certos tokens, repetições etc.). O essencial é o mesmo laço: **encoder uma vez → decoder token a token → argmax**.

> **Atenção:** `generate` com `language="pt"` e `task="transcribe"` equivale a forçar o prefixo `<|startoftranscript|><|pt|><|transcribe|>`. Sem `language`, o modelo detecta o idioma sozinho (seção 5.1). Em versões antigas isso era feito com `forced_decoder_ids`.

## 7. Atenção cruzada: onde o decoder "escuta"?

Cada posição de saída do encoder corresponde a **20 ms** de áudio. Para cada token gerado, a atenção cruzada mostra em quais momentos do áudio o decoder está "olhando". Em geral, o padrão é **quase monotônico** (diagonal): o texto avança junto com o tempo. (Média sobre as cabeças da **última** camada; algumas cabeças são mais nítidas que outras. É um bom ponto para o exercício 3.)

In [ ]:
ids_texto = ids_mao.unsqueeze(0)                                              # prefixo + texto gerado
with torch.no_grad():
    saida = model(encoder_outputs=BaseModelOutput(last_hidden_state=enc_out),
                  decoder_input_ids=ids_texto, output_attentions=True, use_cache=False)

cross = saida.cross_attentions[-1][0].mean(0).cpu()                          # (T, 1500): média das cabeças
quadros = min(1500, int(duracao / 0.02) + 25)                                # só a parte com áudio real
rotulos = tok.convert_ids_to_tokens(ids_texto[0].tolist())

plt.figure(figsize=(9, max(3, 0.28 * len(rotulos))))
plt.imshow(cross[:, :quadros].numpy(), aspect="auto", cmap="viridis", extent=[0, quadros * 0.02, len(rotulos) - 0.5, -0.5])
plt.yticks(range(len(rotulos)), rotulos, fontsize=7)
plt.xlabel("tempo no áudio (s)"); plt.title("Atenção cruzada (última camada do decoder, média das cabeças)")
plt.colorbar(label="peso de atenção"); plt.tight_layout(); plt.show()

## 8. `pipeline` de alto nível e áudios longos

O `pipeline` junta tudo o que fizemos (reamostragem, features, `generate`, decodificação). Para áudios **maiores que 30 s**, ele corta em blocos (`chunk_length_s`) e costura o texto. Com `return_timestamps=True` devolve o trecho de cada frase.

In [ ]:
# (requer o modelo real)
asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL_ID,
    device=DEVICE,
    chunk_length_s=30,
    generate_kwargs={"language": "portuguese", "task": "transcribe"},
)
resultado = asr({"raw": audio, "sampling_rate": SR}, return_timestamps=True)
print(resultado["text"])
for trecho in resultado.get("chunks", []):
    print(trecho["timestamp"], trecho["text"])

## Para discutir em sala

1. Por que o encoder usa posição **senoidal fixa** e o decoder, **aprendida**? (Dica: o comprimento do encoder é sempre 1500; o do decoder varia.)
2. O que muda no treino e na inferência por causa da máscara causal do decoder?
3. Por que o Whisper usa **pré-norm** + LayerNorm final? O que acontece com o gradiente em modelos profundos?
4. Quantos parâmetros estão no embedding? E se o `proj_out` **não** compartilhasse pesos?
5. O que o modelo faz com 5 s de áudio? E com silêncio absoluto? (Teste e comente: alucinações são um risco conhecido.)

## Exercícios

1. Troque `MODEL_ID` por `openai/whisper-base` e compare tempo e qualidade em português.
2. Troque `language` por `"en"` e transcreva o mesmo áudio em português: o que sai? E com `task="translate"`?
3. Plote a atenção cruzada de **cada cabeça** da última camada (em vez da média). Alguma segue o tempo de forma monotônica?
4. Remova `<|notimestamps|>` do prefixo e observe os tokens `<|0.00|>` no resultado.

## Problemas comuns no Windows

| Sintoma | Solução |
|---|---|
| `FileNotFoundError` / `ffmpeg` não encontrado ao abrir `.mp3`/`.m4a` | o ambiente conda já instala `ffmpeg`; confira com `ffmpeg -version` no *Anaconda Prompt* |
| Aviso sobre *symlinks* do Hugging Face | inofensivo (já silenciado); ou ative o Modo Desenvolvedor do Windows |
| Download lento/bloqueado na rede da sala | rode o notebook uma vez antes, com internet: o modelo fica em `C:\Users\<você>\.cache\huggingface` |
| Memória insuficiente com `small`/`medium` | use `tiny`/`base`, ou GPU |
| Transcrição com texto repetido ou inventado | esperado em áudio muito curto/silencioso; use trechos com fala clara |